# W05-A · A search engine for a café menu

A café app has a search box. A customer types *"a cup of tea"*. Which menu items should come first?

A computer can't read, but it can compare numbers. So you will turn every menu item into a **list of numbers**, one
count per word, called a **vector**. Then you measure how alike two vectors are with **cosine similarity**, and give
rare, telling words more weight with **TF-IDF**.

By the end of this notebook you will be able to:
1. list the **vocabulary**: every different word on the menu
2. turn a text into a vector of word counts
3. work out the **cosine similarity** of two vectors
4. search the menu with plain word counts, and see where it goes wrong
5. work out how rare each word is: **document frequency** and **IDF**
6. search again with **TF-IDF** weights
7. find the menu items most like each other
8. see what TF-IDF cannot do

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w05/W05-A-menu-search.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w05")

In [ ]:
# ▶ Setup: just run this cell. These 10 menu items are all the text our search engine will ever read.
import math                       # math.sqrt (square root) and math.log10
from collections import Counter   # Counter: a dictionary that counts things

# Each menu item is one "document": one text the search engine can return.
menu = [
    "a cup of green tea with honey",
    "a glass of iced tea with lemon",
    "a cup of black coffee",
    "a glass of iced coffee with milk",
    "a cup of hot chocolate with cream",
    "a cup of masala chai with ginger",
    "a slice of lemon cake",
    "a slice of chocolate cake with cream",
    "a cup of ginger tea with honey and lemon",
    "a glass of cold coffee with ice cream",
]
print(len(menu), "menu items")

## 1 · The vocabulary

Every vector will have one number per word, so first we need the list of words, always in the same order. The
**vocabulary** is every different word the menu uses, each once, in alphabetical order.

### ✍️ Exercise 1: `vocab`

Collect the words of every item into a set (a set keeps each word once), then sort it into a list.

In [ ]:
words = set()                    # an empty set
for item in menu:
    ...   # ✍️ add the words of this item to the set
vocab = ...   # ✍️ the set, sorted into a list
print(len(vocab), "words:", vocab)
assert len(vocab) == 24 and vocab[0] == "a" and vocab[-1] == "with", "not quite: 24 words, from 'a' to 'with'"
print("✅")

## 2 · A text as a vector

A **vector** here is just a list of numbers: for each word of the vocabulary, in order, how many times it appears in
the text. *"a cup of tea"* has a 1 under *a*, *cup*, *of* and *tea*, and 0 everywhere else.

### ✍️ Exercise 2: `to_vector(text)`

Count the words of the text, then build a list with one count per vocabulary word.

In [ ]:
def to_vector(text):
    counts = Counter(text.split())   # e.g. {"a": 1, "cup": 1, "of": 1, "tea": 1}
    vector = []
    for word in vocab:               # always the same order: the vocabulary's
        ...   # ✍️ add this word's count to the vector
    return vector
print(to_vector("a cup of tea"))
assert len(to_vector("a cup of tea")) == 24, "one number per vocabulary word"
assert sum(to_vector("a cup of tea")) == 4, "four words, so the counts add up to 4"
assert to_vector("tea tea")[vocab.index("tea")] == 2, "a repeated word counts twice"
print("✅")

In [ ]:
# ▶ Just run this cell: every menu item as a vector, one row per item.
count_vectors = []
for item in menu:
    count_vectors.append(to_vector(item))

# Print the vectors like a table. The word names are long, so we show only their first 3 letters.
print(" " * 42, " ".join(w[:3] for w in vocab))
for item, v in zip(menu, count_vectors):        # zip: walk through two lists side by side
    print(f"{item:42s}", "   ".join(str(x) for x in v))

> 📖 **Reading the output.** Each row is one menu item, each column one word. Most numbers are 0: an item uses only a
> few of the 24 words. The columns for *a* and *of* are all 1s: every item has them.

## 3 · Cosine similarity

Think of each vector as an arrow. Two texts are alike when their arrows **point the same way**. Cosine similarity
measures that, in three steps:

- the **dot product**: multiply the two vectors number by number, and add up: a · b = a₁b₁ + a₂b₂ + …
- the **length** of a vector: the square root of the sum of its squares: |a| = √(a₁² + a₂² + …)
- the **cosine**: the dot product divided by both lengths

$$\cos(a, b) = \frac{a \cdot b}{|a| \times |b|}$$

1 means the arrows point the same way; 0 means the texts share no words at all.

### ✍️ Exercise 3: `dot(a, b)`

In [ ]:
def dot(a, b):
    total = 0
    for x, y in zip(a, b):     # the two numbers for the same word
        ...   # ✍️ add x times y to the total
    return total
print(dot([1, 1, 0], [1, 0, 1]))   # 1×1 + 1×0 + 0×1 = 1
assert dot([1, 2, 3], [4, 5, 6]) == 32, "not quite: 4 + 10 + 18 = 32"
print("✅")

### ✍️ Exercise 4: `length(a)`

The length of an arrow, as in Pythagoras: square every number, add them up, take the square root with `math.sqrt`.

In [ ]:
def length(a):
    total = 0
    for x in a:
        ...   # ✍️ add x squared to the total
    return ...   # ✍️ the square root of the total
print(length([3, 4]))   # √(9 + 16) = √25 = 5
assert length([3, 4]) == 5 and length([1, 1, 1, 1]) == 2, "not quite"
print("✅")

### ✍️ Exercise 5: `cosine(a, b)`

Divide the dot product by the two lengths. One catch: a vector of all zeros has length 0, and you can't divide by 0.
Return 0 in that case.

In [ ]:
def cosine(a, b):
    if length(a) == 0 or length(b) == 0:
        return 0                                # an empty text is like nothing
    return ...   # ✍️ the dot product divided by both lengths
# Two words only, tea and coffee, as on the slides: [tea count, coffee count]
print("tea  vs tea tea       :", cosine([1, 0], [2, 0]))            # same direction
print("tea  vs coffee        :", cosine([1, 0], [0, 1]))            # nothing shared
print("tea  vs tea and coffee:", round(cosine([1, 0], [1, 1]), 2))  # in between
assert cosine([1, 0], [2, 0]) == 1 and cosine([1, 0], [0, 1]) == 0, "not quite"
assert cosine([0, 0], [1, 1]) == 0, "an all-zero vector should give 0"
print("✅")

> 📖 **Reading the output.** *tea* and *tea tea* point the same way, so the cosine is 1: saying a word twice makes the
> arrow longer, not different. *tea* and *coffee* share nothing: 0. *tea and coffee* is in between, about 0.71.

## 4 · Search with word counts

To search, turn the query into a vector too, and rank the menu items by their cosine with it. Just run the cell.

In [ ]:
# ▶ Just run this cell.
def search(query, vectors, make_vector, top=4):
    q = make_vector(query)                       # the query as a vector
    scored = []
    for item, v in zip(menu, vectors):
        scored.append((cosine(q, v), item))      # (score, item) pairs
    scored.sort(reverse=True)                    # biggest score first
    for s, item in scored[:top]:                 # the best few
        print(f"  {s:.2f}  {item}")

print("a cup of tea")
search("a cup of tea", count_vectors, to_vector)

> 📖 **Reading the output.** *a cup of green tea with honey* comes first. Good. But *a cup of black coffee* ties with
> *a cup of ginger tea*, and beats *a glass of iced tea*, which is a tea. Why? Coffee shares *a*, *cup* and *of* with
> the query: three of its four words. Those words are on almost every item, so they say nothing about what the
> customer wants.

## 5 · How rare is each word?

The **document frequency** of a word, **df**, is the number of menu items it appears in. *a* is in all 10, *chai* in
just 1.

### ✍️ Exercise 6: `df`

Count, for every word, how many items contain it. Use `set(item.split())`, so a word that appears twice in one item
still counts as one item.

In [ ]:
df = Counter()
for item in menu:
    ...   # ✍️ add 1 for each different word of this item
print(df.most_common(6))   # the 6 most common words, as (word, df) pairs
assert df["a"] == 10 and df["cup"] == 5 and df["chai"] == 1, "not quite"
print("✅")

The **inverse document frequency**, **IDF**, turns df into a weight: big for rare words, 0 for words in every item.

$$\text{idf}(\text{word}) = \log_{10}\frac{N}{\text{df}(\text{word})}$$

N is the number of items, 10. log₁₀ asks *"10 to the power of what gives this number?"*: log₁₀ 10 = 1, log₁₀ 1 = 0.
So a word in 1 of 10 items gets idf 1, a word in all 10 gets idf 0. Python writes it `math.log10(x)`.

### ✍️ Exercise 7: `idf`

Build a dictionary: word → its idf.

In [ ]:
N = len(menu)                              # 10 items
idf = {}                                   # an empty dictionary: word -> idf
for word in vocab:
    idf[word] = ...   # ✍️ log10 of N divided by the word's df
for word in ["a", "of", "with", "cup", "tea", "chai"]:
    print(f"{word:5s} df = {df[word]:2d}   idf = {idf[word]:.2f}")
assert idf["a"] == 0 and idf["chai"] == 1, "not quite: a is in all 10 items, chai in 1"
print("✅")

> 📖 **Reading the output.** *a* and *of* are on every item: idf 0, so they will count for nothing. *with* is on 8:
> almost nothing. *cup* (5 items) gets 0.30, *tea* (3 items) 0.52, and *chai* (1 item) the full 1.

## 6 · Search with TF-IDF

The **TF-IDF** weight of a word in a text is its count in the text (the **term frequency**, tf) times its idf:

$$\text{tf-idf} = \text{count of the word in the text} \times \text{idf}(\text{word})$$

### ✍️ Exercise 8: `to_tfidf(text)`

Like `to_vector`, but each count is multiplied by the word's idf.

In [ ]:
def to_tfidf(text):
    counts = Counter(text.split())
    vector = []
    for word in vocab:
        ...   # ✍️ add this word's count times its idf
    return vector
v = to_tfidf("a cup of tea")
print([round(x, 2) for x in v])
assert abs(v[vocab.index("tea")] - idf["tea"]) < 1e-12 and v[vocab.index("a")] == 0, "not quite"
print("✅")

In [ ]:
# ▶ Just run this cell: the menu as TF-IDF vectors, then the same search as before.
tfidf_vectors = []
for item in menu:
    tfidf_vectors.append(to_tfidf(item))

print("a cup of tea, with counts")
search("a cup of tea", count_vectors, to_vector)
print("a cup of tea, with TF-IDF")
search("a cup of tea", tfidf_vectors, to_tfidf)

> 📖 **Reading the output.** With TF-IDF the three teas come first, and the coffee drops out of the top three. *a* and
> *of* now weigh 0, and *cup* weighs less than *tea*, because more items have a cup than have tea. The scores are
> smaller too: that doesn't matter, only the order does.

### Try your own

Change the query and run the cell again. Try *iced coffee*, *chocolate cake*, *honey*, or a word that is not on the
menu.

In [ ]:
query = "chocolate cake"    # ✍️ try your own
print("counts:")
search(query, count_vectors, to_vector)
print("TF-IDF:")
search(query, tfidf_vectors, to_tfidf)

## 7 · Find similar items

"Customers who look at this item may also like…": for each item, find the other item with the biggest cosine. Just
run the cell.

In [ ]:
# ▶ Just run this cell.
def most_similar(i, vectors):
    best, best_score = None, -1
    for j in range(len(menu)):           # every other item
        if j == i:
            continue                     # skip the item itself
        s = cosine(vectors[i], vectors[j])
        if s > best_score:
            best, best_score = j, s
    return menu[best], best_score

for i, item in enumerate(menu):          # enumerate: the position i and the item together
    c_item, c_score = most_similar(i, count_vectors)
    t_item, t_score = most_similar(i, tfidf_vectors)
    print(item)
    print(f"    counts: {c_score:.2f}  {c_item}")
    print(f"    TF-IDF: {t_score:.2f}  {t_item}")

> 📖 **Reading the output.** Most pairs make sense either way. Look at *a cup of black coffee*: with counts, its
> closest item is *a cup of green tea with honey*, only because both are "a cup of" something. With TF-IDF it is
> *a glass of iced coffee with milk*: another coffee.

## 8 · What TF-IDF cannot see

Just run the cell.

In [ ]:
# ▶ Just run this cell.
print("cold tea")
search("cold tea", tfidf_vectors, to_tfidf, top=3)
print("cosine of 'chai' and 'tea':", cosine(to_tfidf("chai"), to_tfidf("tea")))

> 📖 **Reading the output.** A customer asking for *cold tea* gets *cold coffee* first, above *iced tea*. To TF-IDF,
> *cold* and *iced* are just two different words, and *cold* is the rarer one, so it wins. And *chai* and *tea* have a
> cosine of 0, though chai is a kind of tea. TF-IDF matches
> **words**, not **meanings**. Word vectors, which learn that *cold* is close to *iced*, are the next step.

## Recap

| idea | what you saw |
|---|---|
| Vocabulary | the 24 different words of the menu, in a fixed order |
| Vector | one count per vocabulary word; mostly 0s |
| Cosine similarity | dot product ÷ (length × length); 1 = same direction, 0 = nothing shared |
| Counts alone | *a cup of black coffee* ties with a tea for *"a cup of tea"*, thanks to *a*, *cup*, *of* |
| df and IDF | idf = log₁₀(N ÷ df): 0 for *a* and *of*, 1 for *chai* |
| TF-IDF | count × idf; the teas come first |
| Its limit | *cold* and *iced*, *chai* and *tea*: different words, so no match |